# 🌦️ Climate / Weather Market Backtest

**Purpose:** Clean, production-grade backtest for the Polymarket weather copy-trade strategy.  
This notebook supersedes the exploratory analysis in `practice.ipynb`.

| Section | Contents |
|---|---|
| **0 — Config** | All tunable knobs in one place |
| **1 — Market Universe** | Fetch + classify resolved weather markets with horizon & type filters |
| **2 — Fee Model** | Correct `fee = C × 0.05 × p × (1-p)` formula; `get_fee_rate()` helper |
| **3 — Trade Data** | Fetch BUY trades (taker + maker); apply min-notional & window filter |
| **4 — Signal Generation** | Whale + Consensus copy-trade signals; accuracy gate; seasonal gate; climate stub |
| **5 — Backtest Engine** | Baseline PnL + slippage-adjusted PnL in parallel |
| **6 — Stratified Analysis** | Edge broken down by market type × signal type × season |
| **7 — Walk-Forward** | Expanding-window validation; go-live gate; fee-edge mean & std-dev |
| **8 — Maker Limit Order** | Zero-fee entry modelling; fill-rate assumption |
| **9 — Kelly Sizing** | Fractional Kelly stub (run only after go-live gate passes) |

> ⚠️ Run Section 0 first — every other section imports from the config namespace.

## Section 0 — Configuration

All tunable parameters live here. Re-run this cell to reset state without restarting the kernel.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 0 — CONFIGURATION  (edit here; re-run to reset all parameters)
# ═══════════════════════════════════════════════════════════════════════════════

# ── Data window ───────────────────────────────────────────────────────────────
MONTHS_BACK       = 12     # lookback window in months
SAMPLE_SIZE       = 300    # resolved markets to sample per run (increase for tighter CI)

# ── Market filter ─────────────────────────────────────────────────────────────
RESOLVE_HORIZON_MIN_DAYS = 7    # exclude markets resolving in < 7 days (too short for copy-trade lag)
RESOLVE_HORIZON_MAX_DAYS = 30   # exclude markets resolving in > 30 days (annual/medium-term)
MIN_MARKET_VOLUME        = 1_000  # $ — skip thin markets with < $1K volume
APPLY_FEES_ENABLED_FILTER = True  # skip legacy markets where feesEnabled != True

# ── Fee model (Section 2) ─────────────────────────────────────────────────────
WEATHER_FEE_RATE  = 0.05   # Polymarket weather category feeRate (fetched dynamically per market)
DEFAULT_FEE_RATE  = 0.05   # fallback if CLOB fee-rate API is unavailable

# ── Trade filter ──────────────────────────────────────────────────────────────
MIN_NOTIONAL      = 10     # $ minimum per trade to include
WHALE_NOTIONAL    = 100    # $ threshold for "whale" signal label

# ── Uncertainty band ─────────────────────────────────────────────────────────
ENTRY_LOW         = 0.30   # lower bound of uncertain price range
ENTRY_HIGH        = 0.70   # upper bound of uncertain price range

# ── Signal generation ─────────────────────────────────────────────────────────
CONSENSUS_TRADERS   = 2    # min distinct wallets for a consensus signal
CONSENSUS_WINDOW_H  = 24   # hours window for consensus grouping
TOP_N_WALLETS       = 20   # number of weather leaderboard wallets to track

# ── Accuracy gate (wallet quality filter) ────────────────────────────────────
MIN_WR_GATE    = 0.55   # minimum historical win rate to trust a wallet
MIN_BETS_GATE  = 20     # minimum resolved bets for win rate to be meaningful

# ── Seasonal gate ─────────────────────────────────────────────────────────────
APPLY_SEASONAL_GATE  = True
SEASONAL_GO_MONTHS   = {3, 4, 5, 6, 7, 8}  # Mar–Aug  (spring + summer)

# ── Slippage simulation (Section 5) ──────────────────────────────────────────
SLIPPAGE_PP = 0.02   # percentage points added to whale entry price at your execution
                     # (models the price impact of the whale's own trade before you enter)

# ── API endpoints ─────────────────────────────────────────────────────────────
DATA_BASE   = "https://data-api.polymarket.com"
GAMMA_BASE  = "https://gamma-api.polymarket.com"
CLOB_BASE   = "https://clob.polymarket.com"

# ── HTTP ──────────────────────────────────────────────────────────────────────
REQUEST_TIMEOUT  = 30
MAX_RETRIES      = 3
SLEEP_BETWEEN    = 0.10

# ── Climate forecast signal stub (Section 4 / future stage) ──────────────────
ENABLE_CLIMATE_SIGNAL = False   # set True when NOAA/ECMWF feed is integrated
CLIMATE_SIGNAL_SOURCE = None    # options: "NOAA_API" | "ECMWF" | None

# ── Walk-forward (Section 7) ──────────────────────────────────────────────────
WF_TRAIN_MONTHS    = 9    # months in initial training window
WF_VALIDATE_MONTHS = 1    # months per out-of-sample fold
WF_GO_LIVE_MIN_FE  = 0.0  # mean fee_edge must exceed this to pass go-live gate
WF_GO_LIVE_MIN_POS = 0.75 # fraction of folds that must be positive fee_edge

# ── Maker limit order strategy (Section 8) ────────────────────────────────────
MAKER_TICK_INSIDE  = 0.01   # post limit at best_ask - MAKER_TICK_INSIDE
MAKER_FILL_RATE    = 0.80   # assumed fill rate for maker limit orders (conservative)
MAKER_FEE_RATE     = 0.0    # makers pay zero fees on Polymarket

# ── Kelly sizing (Section 9) ─────────────────────────────────────────────────
KELLY_FRACTION     = 0.25   # fractional Kelly multiplier (25% = safety buffer)
KELLY_MAX_BET_PCT  = 0.05   # never bet > 5% of bankroll per position

import sys, time, random, math, requests, json
from collections import defaultdict
from datetime import datetime, timezone, timedelta
from typing import Any, Dict, List, Optional, Set, Tuple

sys.path.insert(0, ".")
from weather_markets import WeatherMarketFetcher, classify_market_type, classify_resolution_horizon

# Derived constants
NOW_TS    = int(datetime.now(timezone.utc).timestamp())
CUTOFF_TS = int((datetime.now(timezone.utc) - timedelta(days=MONTHS_BACK * 30)).timestamp())

print("=" * 70)
print("CONFIG LOADED")
print("=" * 70)
print(f"  Lookback         : {MONTHS_BACK} months  (from {datetime.fromtimestamp(CUTOFF_TS, tz=timezone.utc).strftime('%Y-%m-%d')})")
print(f"  Market horizon   : {RESOLVE_HORIZON_MIN_DAYS}–{RESOLVE_HORIZON_MAX_DAYS} days  (short-term only)")
print(f"  Uncertainty band : {ENTRY_LOW:.0%}–{ENTRY_HIGH:.0%}")
print(f"  Fee rate         : {WEATHER_FEE_RATE:.0%} (weather category)")
print(f"  Slippage sim     : +{SLIPPAGE_PP*100:.0f}pp")
print(f"  Seasonal gate    : {'ON  ' + str(sorted(SEASONAL_GO_MONTHS)) if APPLY_SEASONAL_GATE else 'OFF'}")
print(f"  Accuracy gate    : WR≥{MIN_WR_GATE:.0%}  n≥{MIN_BETS_GATE}")
print(f"  Climate signal   : {'ENABLED (' + str(CLIMATE_SIGNAL_SOURCE) + ')' if ENABLE_CLIMATE_SIGNAL else 'DISABLED (stub only)'}")
print()
print("  ✅ Run Section 1 next: Market Universe Fetch")

## Section 1 — Market Universe

Fetch all resolved weather markets.  Filter to:
- **Short-term** resolution horizon (7–30 days)
- `feesEnabled = True` (legacy markets excluded)
- Volume ≥ \$1,000

Summarise by **market type** (`temperature`, `event_count`, `index`, `other`) and confirm sample size is adequate for the backtest.

In [ ]:
# ── 1a. Fetch resolved weather markets from Gamma API ─────────────────────────
fetcher  = WeatherMarketFetcher()
raw_list = fetcher.get_weather_markets(resolved=True)

# ── 1b. Apply filters ──────────────────────────────────────────────────────────
filtered: List[Dict[str, Any]] = []
rejected_reasons: Dict[str, int] = defaultdict(int)

for m in raw_list:
    # --- fees gate ---
    if APPLY_FEES_ENABLED_FILTER and not m.get("feesEnabled", False):
        rejected_reasons["feesEnabled=False"] += 1
        continue

    # --- volume gate ---
    vol = float(m.get("volume", 0) or 0)
    if vol < MIN_MARKET_VOLUME:
        rejected_reasons[f"volume<{MIN_MARKET_VOLUME}"] += 1
        continue

    # --- horizon gate ---
    horizon = m.get("resolutionHorizon", "unknown")
    if horizon != "short":
        rejected_reasons[f"horizon={horizon}"] += 1
        continue

    # --- created-at cutoff ---
    created_ts = 0
    raw_created = m.get("createdAt", "")
    if raw_created:
        try:
            created_ts = int(datetime.fromisoformat(raw_created.rstrip("Z")).replace(tzinfo=timezone.utc).timestamp())
        except Exception:
            pass
    if created_ts and created_ts < CUTOFF_TS:
        rejected_reasons["too_old"] += 1
        continue

    filtered.append(m)

print(f"Raw markets fetched  : {len(raw_list):>6}")
print(f"Markets after filters: {len(filtered):>6}")
print()
print("Rejection breakdown:")
for reason, cnt in sorted(rejected_reasons.items(), key=lambda x: -x[1]):
    print(f"  {reason:<30} {cnt:>5}")

# ── 1c. Market-type distribution ──────────────────────────────────────────────
type_counts: Dict[str, int] = defaultdict(int)
for m in filtered:
    type_counts[m.get("marketType", "other")] += 1

print()
print("Market-type distribution (filtered universe):")
for mtype, cnt in sorted(type_counts.items(), key=lambda x: -x[1]):
    bar = "█" * int(cnt / max(type_counts.values()) * 30)
    print(f"  {mtype:<15} {cnt:>4}  {bar}")

# ── 1d. Sample if needed ──────────────────────────────────────────────────────
if len(filtered) > SAMPLE_SIZE:
    random.seed(42)
    markets = random.sample(filtered, SAMPLE_SIZE)
    print(f"\nSampled {SAMPLE_SIZE} of {len(filtered)} markets (seed=42).")
else:
    markets = filtered
    print(f"\nUsing all {len(markets)} markets.")

MARKETS = markets  # canonical list for downstream sections

## Section 2 — Fee Model

Polymarket charges a **non-flat fee** on weather markets:

$$\text{fee} = C \times r \times p \times (1-p)$$

where $C$ = shares purchased, $r$ = fee rate (0.05 for weather), and $p$ = entry price in \[0,1\].

As a percentage of notional spent ($C \times p$):

$$\text{fee\%} = r \times (1 - p)$$

| Entry price $p$ | Old flat model | Correct model | Δ |
|---|---|---|---|
| 0.30 | 2.0% | 3.5% | +1.5pp |
| 0.50 | 2.0% | 2.5% | +0.5pp |
| 0.70 | 2.0% | 1.5% | −0.5pp |

The old model undercounted fees for cheap outcomes and overcounted for expensive ones.  The correct model adjusts the **hurdle rate** differently depending on which side of 50¢ you trade.

In [ ]:
# ── 2a. Core fee functions ─────────────────────────────────────────────────────

def compute_fee(shares: float, price: float, fee_rate: float = DEFAULT_FEE_RATE) -> float:
    """Return dollar fee charged on a taker BUY of `shares` contracts at `price`.

    Formula: fee = shares * fee_rate * price * (1 - price)
    Source:  Polymarket docs — 'feeAmount = size * feeRate * price * (1 - price)'
    """
    return shares * fee_rate * price * (1.0 - price)


def fee_pct_of_notional(price: float, fee_rate: float = DEFAULT_FEE_RATE) -> float:
    """Return fee as a fraction of notional spent (shares * price).

    fee / notional = fee_rate * (1 - price)
    """
    return fee_rate * (1.0 - price)


def fee_breakeven_wr(price: float, fee_rate: float = DEFAULT_FEE_RATE) -> float:
    """Win rate needed to break even net of fees when buying YES at `price`.

    If you win you collect $1/share; entry cost = price + fee/share.
    BE win rate = (price + fee/share) / 1.0
               = price + fee_rate * price * (1 - price)
    """
    return price + fee_rate * price * (1.0 - price)


def get_fee_rate(token_id: str, timeout: int = REQUEST_TIMEOUT) -> float:
    """Query CLOB API for the fee rate of a specific market token.

    Returns the float fee rate (e.g. 0.05) or DEFAULT_FEE_RATE on error.
    stub — uncomment when CLOB keys are configured.
    """
    # try:
    #     url = f"{CLOB_BASE}/fee-rate?token_id={token_id}"
    #     resp = requests.get(url, timeout=timeout)
    #     resp.raise_for_status()
    #     data = resp.json()
    #     return float(data.get("fee_rate", DEFAULT_FEE_RATE))
    # except Exception as exc:
    #     print(f"  [WARN] get_fee_rate({token_id!r}) failed: {exc}")
    #     return DEFAULT_FEE_RATE
    return DEFAULT_FEE_RATE   # stub — replace with live call when keys available


# ── 2b. Comparison table: old flat model vs correct model ─────────────────────
SAMPLE_PRICES = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]
OLD_RATE = 0.02   # flat rate used in practice.ipynb

print(f"{'Entry p':>9}  {'Old flat%':>10}  {'Correct%':>10}  {'Δ pp':>8}  {'BE win rate':>12}")
print("-" * 58)
for p in SAMPLE_PRICES:
    old_pct = OLD_RATE
    new_pct = fee_pct_of_notional(p, WEATHER_FEE_RATE)
    delta   = (new_pct - old_pct) * 100
    be_wr   = fee_breakeven_wr(p)
    print(f"  p={p:.2f}    {old_pct*100:>8.2f}%  {new_pct*100:>8.2f}%  {delta:>+7.2f}pp  {be_wr:>10.4f}")

print()
print("Interpretation:")
print("  • For cheap outcomes (p < 0.60) the old model UNDERCOUNTED fees — ")
print("    live edge is lower than practice.ipynb suggested.")
print("  • For expensive outcomes (p > 0.60) the old model OVERCOUNTED fees.")
print("  • Biggest correction is at p=0.30:  actual fee 3.5% vs old 2.0% (+1.5pp).")
print()
print(f"FEE MODEL SET — rate: {WEATHER_FEE_RATE:.0%}  |  formula: shares × r × p × (1-p)")

## Section 3 — Trade Data

Fetch taker BUY trades for each market.  Key fixes vs `practice.ipynb`:

1. **`takerOnly=false`** — the API interprets `true` as "only return taker-side records", which works fine; but the Polymarket CLOB API sometimes drops records when this param is unexpected.  We query without it and apply a **client-side `side=BUY` filter** so we never miss trades.
2. **Retry with backoff** — `_safe_get()` wraps every request with up to `MAX_RETRIES` attempts.
3. Each trade record is enriched with `marketType` and `resolutionHorizon` from the parent market.

In [ ]:
# ── 3a. HTTP helpers ──────────────────────────────────────────────────────────

def _safe_get(url: str, params: Optional[Dict] = None, timeout: int = REQUEST_TIMEOUT,
              max_retries: int = MAX_RETRIES, sleep_s: float = SLEEP_BETWEEN) -> Optional[Any]:
    """GET with exponential-backoff retry.  Returns parsed JSON or None on failure."""
    for attempt in range(1, max_retries + 1):
        try:
            resp = requests.get(url, params=params, timeout=timeout)
            resp.raise_for_status()
            return resp.json()
        except Exception as exc:
            wait = sleep_s * (2 ** (attempt - 1))
            if attempt < max_retries:
                time.sleep(wait)
            else:
                print(f"  [WARN] {url} failed after {max_retries} attempts: {exc}")
                return None


# ── 3b. Fetch trades per market ───────────────────────────────────────────────

def fetch_trades_for_market(market: Dict[str, Any]) -> List[Dict[str, Any]]:
    """Return all resolved taker BUY trades for a market.

    BUG-6 fix: query without takerOnly=true; filter client-side by side='BUY'.
    """
    token_ids: List[str] = []
    for tok in market.get("tokens", []):
        if isinstance(tok, dict):
            tid = tok.get("token_id") or tok.get("id") or ""
            if tid:
                token_ids.append(str(tid))

    if not token_ids:
        return []

    trades: List[Dict[str, Any]] = []
    for tid in token_ids:
        url    = f"{DATA_BASE}/trades"
        params = {"market": tid, "limit": 500}
        data   = _safe_get(url, params=params)

        if not data:
            continue

        raw_trades = data if isinstance(data, list) else data.get("data", [])
        for t in raw_trades:
            # ── client-side BUY filter (BUG-6 fix) ──────────────────────────
            if str(t.get("side", "")).upper() != "BUY":
                continue

            notional = float(t.get("size", 0) or 0) * float(t.get("price", 0) or 0)
            if notional < MIN_NOTIONAL:
                continue

            # ── enrich with market metadata ───────────────────────────────────
            t["_market_id"]          = market.get("conditionId") or market.get("id")
            t["_market_type"]        = market.get("marketType", "other")
            t["_resolution_horizon"] = market.get("resolutionHorizon", "unknown")
            t["_outcome_resolved"]   = market.get("resolved_outcome")
            t["_question"]           = market.get("question", "")
            t["_end_date"]           = market.get("endDate", "")
            trades.append(t)

    return trades


# ── 3c. Build full trade dataset ─────────────────────────────────────────────

all_trades: List[Dict[str, Any]] = []
errors = 0

for i, mkt in enumerate(MARKETS):
    batch = fetch_trades_for_market(mkt)
    all_trades.extend(batch)
    time.sleep(SLEEP_BETWEEN)
    if (i + 1) % 50 == 0:
        print(f"  [{i+1}/{len(MARKETS)}] trades so far: {len(all_trades)}")

print(f"\nTotal trades fetched : {len(all_trades):>6}")

# ── 3d. Quick summary ────────────────────────────────────────────────────────
mtype_trade_counts: Dict[str, int] = defaultdict(int)
for t in all_trades:
    mtype_trade_counts[t.get("_market_type", "other")] += 1

print("\nTrades by market type:")
for mt, cnt in sorted(mtype_trade_counts.items(), key=lambda x: -x[1]):
    print(f"  {mt:<15} {cnt:>6}")

TRADES = all_trades  # canonical list for downstream sections

## Section 4 — Signal Generation

Three layered signal types, each progressively more filtered:

| Layer | Signal | Description |
|---|---|---|
| **L1** | Whale | Single wallet spends ≥ \$100 on one side |
| **L2** | Consensus | ≥ 2 distinct wallets all buy the same side within 24 h |
| **L3** | Gated | L1 or L2 from a wallet that passes the accuracy gate (WR ≥ 55%, n ≥ 20) |

An optional **seasonal gate** restricts analysis to spring/summer months (Mar–Aug).

A **climate forecast stub** slot is reserved for future NOAA/ECMWF integration.

In [ ]:
# ── 4a. Load leaderboard wallets (accuracy gate) ─────────────────────────────

def fetch_top_wallets(n: int = TOP_N_WALLETS) -> Dict[str, Dict[str, Any]]:
    """Fetch the top-N weather traders from the leaderboard API.

    Returns a dict keyed by maker_address with {win_rate, total_bets}.
    """
    url  = f"{DATA_BASE}/leaderboard"
    data = _safe_get(url, params={"tags": "weather", "limit": n * 2}) or {}
    rows = data if isinstance(data, list) else data.get("data", [])
    wallets: Dict[str, Dict[str, Any]] = {}
    for row in rows:
        addr = row.get("proxyWalletAddress") or row.get("userAddress") or ""
        if not addr:
            continue
        wins  = int(row.get("numWins",  0) or 0)
        total = int(row.get("numBets",  0) or row.get("totalTrades", 0) or 0)
        if total > 0:
            wallets[addr.lower()] = {
                "win_rate":   wins / total,
                "total_bets": total,
            }
    return dict(list(wallets.items())[:n])


LEADERBOARD_WALLETS = fetch_top_wallets()
print(f"Leaderboard wallets loaded: {len(LEADERBOARD_WALLETS)}")

# ── 4b. Accuracy gate ─────────────────────────────────────────────────────────

def passes_accuracy_gate(maker: str) -> bool:
    """Return True if the wallet meets quality thresholds."""
    info = LEADERBOARD_WALLETS.get(maker.lower(), {})
    return (
        info.get("win_rate", 0)   >= MIN_WR_GATE and
        info.get("total_bets", 0) >= MIN_BETS_GATE
    )

# ── 4c. Seasonal gate ─────────────────────────────────────────────────────────

def passes_seasonal_gate(ts_str: str) -> bool:
    """Return True if the trade timestamp falls in a go-live month."""
    if not APPLY_SEASONAL_GATE:
        return True
    try:
        ts = int(ts_str)
        month = datetime.fromtimestamp(ts, tz=timezone.utc).month
        return month in SEASONAL_GO_MONTHS
    except Exception:
        return True   # permissive on parse errors

# ── 4d. Climate forecast stub ────────────────────────────────────────────────
# TODO: Replace with live NOAA/ECMWF forecast lookup when ENABLE_CLIMATE_SIGNAL=True.
# The function should return the forecast probability for the YES outcome
# based on the market question and resolution date.

def climate_signal_override(question: str, end_date: str) -> Optional[float]:
    """Return a NOAA/ECMWF-derived probability for the YES outcome, or None.

    Stub: always returns None until ENABLE_CLIMATE_SIGNAL is True and
    CLIMATE_SIGNAL_SOURCE is configured.
    """
    if not ENABLE_CLIMATE_SIGNAL:
        return None
    # ── future integration point ─────────────────────────────────────────────
    # if CLIMATE_SIGNAL_SOURCE == "NOAA_API":
    #     return _fetch_noaa_probability(question, end_date)
    # elif CLIMATE_SIGNAL_SOURCE == "ECMWF":
    #     return _fetch_ecmwf_probability(question, end_date)
    return None

# ── 4e. Generate signals from trade data ─────────────────────────────────────

def _parse_ts(val: Any) -> int:
    try:
        return int(val)
    except Exception:
        return 0


def generate_signals(trades: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    Tag each qualifying trade with signal type, accounting for:
    - L1: whale (single notional ≥ WHALE_NOTIONAL)
    - L2: consensus (≥ CONSENSUS_TRADERS distinct makers on same side within window)
    - L3: gated (L1|L2 from quality wallet)
    - seasonal gate
    - entry price in uncertainty band [ENTRY_LOW, ENTRY_HIGH]
    Returns a list of signal records.
    """
    # Group by market for consensus detection
    by_market: Dict[str, List[Dict]] = defaultdict(list)
    for t in trades:
        by_market[t.get("_market_id", "")].append(t)

    signals: List[Dict[str, Any]] = []

    for mkt_id, mkt_trades in by_market.items():
        # Sort by timestamp
        mkt_trades_sorted = sorted(mkt_trades, key=lambda t: _parse_ts(t.get("timestamp", 0)))

        for i, t in enumerate(mkt_trades_sorted):
            price = float(t.get("price", 0) or 0)
            if not (ENTRY_LOW <= price <= ENTRY_HIGH):
                continue

            ts       = _parse_ts(t.get("timestamp", 0))
            notional = float(t.get("size", 0) or 0) * price
            maker    = (t.get("maker") or t.get("makerAddress") or "").lower()

            # Seasonal gate
            if not passes_seasonal_gate(str(ts)):
                continue

            # ── L1: whale signal ────────────────────────────────────────────
            is_whale = notional >= WHALE_NOTIONAL

            # ── L2: consensus signal ────────────────────────────────────────
            window_start = ts - CONSENSUS_WINDOW_H * 3600
            side = str(t.get("side", "BUY")).upper()
            nearby_makers: Set[str] = set()
            for other in mkt_trades_sorted:
                other_ts = _parse_ts(other.get("timestamp", 0))
                if other_ts < window_start or other_ts > ts:
                    continue
                if str(other.get("side", "BUY")).upper() != side:
                    continue
                om = (other.get("maker") or other.get("makerAddress") or "").lower()
                if om:
                    nearby_makers.add(om)
            is_consensus = len(nearby_makers) >= CONSENSUS_TRADERS

            if not (is_whale or is_consensus):
                continue

            # ── L3: gated ──────────────────────────────────────────────────
            is_gated = (is_whale or is_consensus) and passes_accuracy_gate(maker)

            # ── Climate forecast override ───────────────────────────────────
            climate_p = climate_signal_override(
                t.get("_question", ""), t.get("_end_date", "")
            )

            signals.append({
                "market_id":   mkt_id,
                "market_type": t["_market_type"],
                "horizon":     t["_resolution_horizon"],
                "ts":          ts,
                "maker":       maker,
                "price":       price,
                "notional":    notional,
                "resolved":    t.get("_outcome_resolved"),
                "is_whale":    is_whale,
                "is_consensus": is_consensus,
                "is_gated":    is_gated,
                "climate_p":  climate_p,
                "signal_type": (
                    "gated"     if is_gated     else
                    "consensus" if is_consensus else
                    "whale"
                ),
            })

    return signals


SIGNALS = generate_signals(TRADES)

# ── 4f. Summary ───────────────────────────────────────────────────────────────
stype_counts: Dict[str, int] = defaultdict(int)
for s in SIGNALS:
    stype_counts[s["signal_type"]] += 1

print(f"Total signals generated: {len(SIGNALS)}")
print()
for st, cnt in [("whale", stype_counts["whale"]),
                ("consensus", stype_counts["consensus"]),
                ("gated", stype_counts["gated"])]:
    print(f"  {st:<12}: {cnt:>5}")
print()
print(f"  ⚠ Climate signal: {'ACTIVE' if ENABLE_CLIMATE_SIGNAL else 'STUB ONLY (disabled)'}")

## Section 5 — Backtest Engine

Two parallel evaluation passes on every signal:

| Pass | Entry price | Purpose |
|---|---|---|
| **Baseline** | whale's `price` | Upper-bound performance — assumes instant fill at whale's price |
| **Slippage** | `min(price + 0.02, 0.99)` | Realistic — you enter *after* the whale, price has moved +2pp |

Key metrics per pass:
- **win_rate** — fraction of signals where `resolved == "YES"`
- **avg_p** — mean entry price
- **fee_edge** — `win_rate − avg_p − fee_rate × avg_p × (1 − avg_p)`
- **95% bootstrap CI** on win_rate (2 000 resamples)

In [ ]:
# ── 5a. Bootstrap CI helper ──────────────────────────────────────────────────

def bootstrap_wr_ci(outcomes: List[int], n_boot: int = 2000, alpha: float = 0.05
                    ) -> Tuple[float, float]:
    """Return (ci_lo, ci_hi) for win rate via percentile bootstrap."""
    if not outcomes:
        return (float("nan"), float("nan"))
    wins = sum(outcomes)
    n    = len(outcomes)
    boot: List[float] = []
    for _ in range(n_boot):
        sample = sum(random.choices(outcomes, k=n))
        boot.append(sample / n)
    boot.sort()
    lo_idx = int(math.floor(alpha / 2 * n_boot))            # 50th element (0-indexed) → 2.5%
    hi_idx = int(math.ceil((1 - alpha / 2) * n_boot)) - 1   # 1950th element            → 97.5%
    return (boot[lo_idx], boot[hi_idx])


# ── 5b. Single-pass summarise ─────────────────────────────────────────────────

def _summarise(records: List[Dict[str, Any]], label: str,
               fee_rate: float = WEATHER_FEE_RATE) -> Dict[str, Any]:
    """Compute backtest statistics for a list of signal records."""
    if not records:
        return {"label": label, "n": 0}

    prices   = [r["entry_price"] for r in records]
    outcomes = []
    for r in records:
        res = r.get("resolved")
        if res is not None:
            outcomes.append(1 if str(res).upper() in ("YES", "1", "TRUE") else 0)

    n        = len(outcomes)
    if n == 0:
        return {"label": label, "n": 0, "note": "no resolved outcomes"}

    wr       = sum(outcomes) / n
    avg_p    = sum(prices[:n]) / n
    fee_edge = wr - avg_p - fee_rate * avg_p * (1.0 - avg_p)
    ci_lo, ci_hi = bootstrap_wr_ci(outcomes)

    return {
        "label":     label,
        "n":         n,
        "win_rate":  wr,
        "avg_p":     avg_p,
        "fee_edge":  fee_edge,
        "ci_lo":     ci_lo,
        "ci_hi":     ci_hi,
        "ci_width":  ci_hi - ci_lo,
        "positive":  fee_edge > 0,
    }


# ── 5c. Two-pass evaluation ───────────────────────────────────────────────────

def run_backtest_two_pass(signals: List[Dict[str, Any]],
                          fee_rate: float = WEATHER_FEE_RATE,
                          signal_filter: Optional[str] = None,
                          ) -> Tuple[Dict, Dict]:
    """
    Run baseline and slippage-adjusted backtest passes.

    signal_filter: if provided, restrict to signals where signal_type == this value.
    Returns (baseline_stats, slippage_stats).
    """
    subset = [s for s in signals
              if signal_filter is None or s["signal_type"] == signal_filter]

    baseline_records  = [{"entry_price": s["price"],   "resolved": s["resolved"]} for s in subset]
    slippage_records  = [{"entry_price": min(s["price"] + SLIPPAGE_PP, 0.99),
                          "resolved":    s["resolved"]} for s in subset]

    label  = signal_filter or "all"
    return (
        _summarise(baseline_records,  f"{label} [baseline]",  fee_rate),
        _summarise(slippage_records,  f"{label} [slippage +{SLIPPAGE_PP*100:.0f}pp]", fee_rate),
    )


# ── 5d. Print results ─────────────────────────────────────────────────────────

def _print_stats(stats: Dict[str, Any]) -> None:
    n = stats.get("n", 0)
    if n == 0:
        print(f"  {stats['label']:<40}  n=0")
        return
    fe   = stats["fee_edge"]
    flag = "✅" if stats.get("positive") else "❌"
    print(
        f"  {stats['label']:<40}  "
        f"n={n:<5}  WR={stats['win_rate']:.1%}  "
        f"avg_p={stats['avg_p']:.3f}  "
        f"fee_edge={fe:+.1%}  "
        f"CI95=[{stats['ci_lo']:.3f},{stats['ci_hi']:.3f}]  {flag}"
    )


print("=" * 100)
print("BACKTEST  —  ALL SIGNALS  (two passes)")
print("=" * 100)
for stype in [None, "whale", "consensus", "gated"]:
    base, slip = run_backtest_two_pass(SIGNALS, signal_filter=stype)
    _print_stats(base)
    _print_stats(slip)
    print()

# ── 5e. Store full results for downstream sections ────────────────────────────
BACKTEST_RESULTS: Dict[str, Tuple[Dict, Dict]] = {}
for stype in ["whale", "consensus", "gated"]:
    BACKTEST_RESULTS[stype] = run_backtest_two_pass(SIGNALS, signal_filter=stype)

## Section 6 — Stratified Analysis

Break down performance along three dimensions simultaneously:

- **Market type**: `temperature` / `event_count` / `index` / `other`
- **Signal type**: `whale` / `consensus` / `gated`
- **Season**: go-months (Mar–Aug) vs off-months (Sep–Feb)

Each cell shows `fee_edge` (slippage-adjusted) and sample size.  Cells with fewer than 10 observations are marked *(sparse)*.

In [ ]:
# ── 6a. Helpers ──────────────────────────────────────────────────────────────

def _season_label(ts: int) -> str:
    try:
        month = datetime.fromtimestamp(ts, tz=timezone.utc).month
        return "go-months" if month in SEASONAL_GO_MONTHS else "off-months"
    except Exception:
        return "unknown"


SPARSE_THRESHOLD = 10

def _cell_str(stats: Dict[str, Any]) -> str:
    n = stats.get("n", 0)
    if n == 0:
        return "  —  "
    tag    = "*(sparse)*" if n < SPARSE_THRESHOLD else ""
    fe_str = f"{stats['fee_edge']:+.1%}"
    slip_base, slip_slip = run_backtest_two_pass(
        [],   # already summarised — use stored value
    ) if False else (stats, stats)
    return f"{fe_str} n={n} {tag}"


# ── 6b. Build stratification grid ────────────────────────────────────────────

MARKET_TYPES  = ["temperature", "event_count", "index", "other"]
SIGNAL_TYPES  = ["whale", "consensus", "gated"]
SEASON_LABELS = ["go-months", "off-months"]

# Bucket signals
grid: Dict[Tuple, List[Dict]] = defaultdict(list)
for s in SIGNALS:
    season = _season_label(s["ts"])
    key    = (s["market_type"], s["signal_type"], season)
    grid[key].append(s)


# ── 6c. Print stratification table ───────────────────────────────────────────

def _fmt_fe(signals: List[Dict], stype: str, season: str, mtype: str) -> str:
    subset = [s for s in signals
              if s["signal_type"] == stype
              and _season_label(s["ts"]) == season
              and s["market_type"] == mtype]
    if not subset:
        return "  —  "
    _, slip_stats = run_backtest_two_pass(subset)
    n  = slip_stats.get("n", 0)
    if n == 0:
        return "  —  "
    fe = slip_stats["fee_edge"]
    tag = " ⚠" if n < SPARSE_THRESHOLD else ""
    return f"{fe:+.1%}(n={n}){tag}"


print("Stratified fee_edge  [slippage-adjusted]")
print("(⚠ = fewer than 10 observations — treat with caution)")
print()

COL_W = 22

for season in SEASON_LABELS:
    print(f"  ── {season.upper()} ──────────────────────────────────────────────────────────")
    header = f"  {'market_type':<15}" + "".join(f"  {st:<{COL_W}}" for st in SIGNAL_TYPES)
    print(header)
    print("  " + "─" * (15 + (COL_W + 2) * len(SIGNAL_TYPES)))
    for mtype in MARKET_TYPES:
        row = f"  {mtype:<15}"
        for stype in SIGNAL_TYPES:
            row += f"  {_fmt_fe(SIGNALS, stype, season, mtype):<{COL_W}}"
        print(row)
    print()

# ── 6d. Top performing strata ────────────────────────────────────────────────
print("\nTop 5 strata by slippage-adjusted fee_edge:")
strata_rows = []
for mtype in MARKET_TYPES:
    for stype in SIGNAL_TYPES:
        for season in SEASON_LABELS:
            subset = [s for s in SIGNALS
                      if s["market_type"] == mtype
                      and s["signal_type"] == stype
                      and _season_label(s["ts"]) == season]
            if not subset:
                continue
            _, slip = run_backtest_two_pass(subset)
            if slip.get("n", 0) >= SPARSE_THRESHOLD:
                strata_rows.append((slip["fee_edge"], mtype, stype, season, slip["n"]))

strata_rows.sort(reverse=True)
for fe, mtype, stype, season, n in strata_rows[:5]:
    print(f"  {fe:+.1%}  {mtype:<15} × {stype:<12} × {season}  (n={n})")

## Section 7 — Expanding Walk-Forward Validation

Simulate how the strategy would have been approved for live trading using a **purged expanding walk-forward** procedure:

- **Fold 1**: train months 1–9, validate month 10
- **Fold 2**: train months 1–10, validate month 11
- … and so on until data runs out

The training window only informs which wallets to include (accuracy gate recalculated in-sample per fold); the test statistic is the **out-of-sample fee_edge** in the hold-out month.

### Go-live gate
The strategy passes if **all** of:
1. Mean out-of-sample fee_edge > 0
2. ≥ 75% of folds have positive fee_edge

If either condition fails, output a `HOLD — do not go live` warning.

In [ ]:
# ── 7a. Assign each signal a calendar month ──────────────────────────────────

def _month_key(ts: int) -> Tuple[int, int]:
    dt = datetime.fromtimestamp(ts, tz=timezone.utc)
    return (dt.year, dt.month)


# Build chronologically sorted list of unique months present in SIGNALS
months_present = sorted({_month_key(s["ts"]) for s in SIGNALS if s["ts"]})

print(f"Months present in signal dataset: {len(months_present)}")
for ym in months_present:
    n_in_month = sum(1 for s in SIGNALS if _month_key(s["ts"]) == ym)
    print(f"  {ym[0]}-{ym[1]:02d}  n={n_in_month}")


# ── 7b. Expanding walk-forward ────────────────────────────────────────────────

MIN_TRAIN_MONTHS = WF_TRAIN_MONTHS

fold_results: List[Dict[str, Any]] = []

for val_idx in range(MIN_TRAIN_MONTHS, len(months_present)):
    train_months = set(months_present[:val_idx])
    val_month    = months_present[val_idx]

    train_sigs = [s for s in SIGNALS if _month_key(s["ts"]) in train_months]
    val_sigs   = [s for s in SIGNALS if _month_key(s["ts"]) == val_month]

    if not val_sigs:
        continue

    # Recalculate accuracy-gate wallets IN-SAMPLE (prevent lookahead)
    oos_wallets: Dict[str, Dict] = defaultdict(lambda: {"wins": 0, "n": 0})
    for ts in train_sigs:
        mk = ts.get("maker", "")
        if not mk:
            continue
        oos_wallets[mk]["n"] += 1
        if str(ts.get("resolved", "")).upper() in ("YES", "1", "TRUE"):
            oos_wallets[mk]["wins"] += 1

    trusted_in_fold: Set[str] = {
        mk for mk, d in oos_wallets.items()
        if d["n"] >= MIN_BETS_GATE and (d["wins"] / d["n"]) >= MIN_WR_GATE
    }

    # Restrict val signals to trusted-in-fold wallets for "gated" signal type
    def _gated_in_fold(s: Dict) -> bool:
        return s.get("maker", "") in trusted_in_fold

    val_all    = val_sigs
    val_gated  = [s for s in val_sigs if _gated_in_fold(s)]
    val_whale  = [s for s in val_sigs if s["is_whale"]]

    # Slippage-adjusted stats for this fold
    _, slip_all   = run_backtest_two_pass(val_all,   signal_filter=None)
    _, slip_gated = run_backtest_two_pass(val_gated, signal_filter=None)
    _, slip_whale = run_backtest_two_pass(val_whale,  signal_filter=None)

    fold_results.append({
        "val_month":      f"{val_month[0]}-{val_month[1]:02d}",
        "train_months":   val_idx,
        "n_val":          slip_all.get("n", 0),
        "fe_all":         slip_all.get("fee_edge", float("nan")),
        "fe_gated":       slip_gated.get("fee_edge", float("nan")),
        "fe_whale":       slip_whale.get("fee_edge", float("nan")),
        "trusted_wallets": len(trusted_in_fold),
    })


# ── 7c. Print fold results ────────────────────────────────────────────────────

print("\nWalk-forward results  [slippage-adjusted fee_edge]")
print(f"  {'Fold':>6}  {'Train mo':>8}  {'n_val':>6}  {'fe_all':>8}  {'fe_gated':>8}  {'fe_whale':>8}  {'trusted_w':>9}")
print("  " + "─" * 65)

fe_all_list: List[float] = []
for r in fold_results:
    fe = r["fe_all"]
    fe_all_list.append(fe)
    flag = "✅" if fe > 0 else "❌"
    print(
        f"  {r['val_month']:>6}  {r['train_months']:>8}  "
        f"{r['n_val']:>6}  "
        f"{r['fe_all']:>+7.1%}  "
        f"{r['fe_gated']:>+7.1%}  "
        f"{r['fe_whale']:>+7.1%}  "
        f"{r['trusted_wallets']:>9}  {flag}"
    )

# ── 7d. Go-live gate ──────────────────────────────────────────────────────────
valid_fes = [f for f in fe_all_list if not math.isnan(f)]
if valid_fes:
    mean_fe     = sum(valid_fes) / len(valid_fes)
    frac_pos    = sum(1 for f in valid_fes if f > 0) / len(valid_fes)
    gate_passed = (mean_fe > WF_GO_LIVE_MIN_FE) and (frac_pos >= WF_GO_LIVE_MIN_POS)

    print()
    print("=" * 65)
    print("GO-LIVE GATE ASSESSMENT")
    print("=" * 65)
    print(f"  Mean OOS fee_edge : {mean_fe:+.2%}  (threshold: >{WF_GO_LIVE_MIN_FE:.0%})")
    print(f"  Fraction positive : {frac_pos:.0%}         (threshold: ≥{WF_GO_LIVE_MIN_POS:.0%})")
    print()
    if gate_passed:
        print("  🟢  GO — both conditions met.  Strategy is cleared for paper trading.")
    else:
        print("  🔴  HOLD — one or more conditions failed.  Do NOT go live.")
        if mean_fe <= WF_GO_LIVE_MIN_FE:
            print("      ↳ Mean fee_edge is non-positive.")
        if frac_pos < WF_GO_LIVE_MIN_POS:
            print(f"      ↳ Only {frac_pos:.0%} of folds were positive (need ≥{WF_GO_LIVE_MIN_POS:.0%}).")
else:
    print("\n  ⚠ Not enough data for walk-forward evaluation.")

## Section 8 — Maker Limit Order Strategy

**Key insight**: on Polymarket, *makers* (limit orders) pay **zero fees**.  Only takers pay the `C × r × p × (1-p)` charge.

Strategy: instead of chasing the whale's fill price, **post a limit order one tick inside the spread** (`best_ask − 0.01`).  If filled you entered at a better price *and* paid no fee — double benefit.

| | Taker (current) | Maker (limit) |
|---|---|---|
| Entry price | whale_price + 2pp slippage | best_ask − 1pp |
| Fee | `r × p × (1-p)` notional | **0** |
| Fill certainty | ~100% | ~80% (assumed) |

This cell models the maker strategy on the existing signal set assuming 80% fill rate.

In [ ]:
# ── 8a. Maker limit order simulation ─────────────────────────────────────────

def simulate_maker_strategy(signals: List[Dict[str, Any]],
                             fill_rate: float = MAKER_FILL_RATE,
                             tick_inside: float = MAKER_TICK_INSIDE,
                             fee_rate: float = MAKER_FEE_RATE,
                             ) -> Dict[str, Any]:
    """
    Model maker limit entry:
    - Entry price = whale_price - tick_inside  (better than whale's price)
    - Fee = 0  (maker pays no fee)
    - Only fill_rate fraction of signals are assumed to get filled

    Returns summary stats comparable to _summarise() output.
    """
    random.seed(42)   # reproducible fill sampling
    filled = []
    for s in signals:
        if random.random() > fill_rate:
            continue   # unfilled — skip
        maker_entry = max(s["price"] - tick_inside, 0.01)
        filled.append({
            "entry_price": maker_entry,
            "resolved":    s["resolved"],
        })

    if not filled:
        return {"label": "maker", "n": 0}

    prices   = [r["entry_price"] for r in filled]
    outcomes = [1 if str(r["resolved"]).upper() in ("YES", "1", "TRUE") else 0
                for r in filled if r["resolved"] is not None]

    n      = len(outcomes)
    if n == 0:
        return {"label": "maker", "n": 0}

    wr     = sum(outcomes) / n
    avg_p  = sum(prices[:n]) / n
    # maker fee = 0, so fee_edge = WR - avg_p (no fee drag)
    fe     = wr - avg_p
    ci_lo, ci_hi = bootstrap_wr_ci(outcomes)

    return {
        "label":         f"maker limit  [{fill_rate:.0%} fill]",
        "n_attempted":   len(signals),
        "n_filled":      len(filled),
        "fill_rate":     fill_rate,
        "n":             n,
        "win_rate":      wr,
        "avg_p":         avg_p,
        "fee_edge":      fe,   # zero fee — full edge is win_rate - avg_p
        "ci_lo":         ci_lo,
        "ci_hi":         ci_hi,
        "positive":      fe > 0,
    }


# ── 8b. Run for each signal type ──────────────────────────────────────────────

print("=" * 90)
print("MAKER LIMIT ORDER STRATEGY  (fee=0, fill_rate=80%)")
print("=" * 90)
print()
print("Compare each row with the equivalent taker slippage row from Section 5.")
print()
print(f"  {'Signal type':<12}  {'attempted':>10}  {'filled':>8}  "
      f"{'WR':>6}  {'avg_p':>7}  {'fee_edge':>9}  "
      f"{'CI95':>17}  ")
print("  " + "─" * 80)

for stype in ["whale", "consensus", "gated"]:
    subset = [s for s in SIGNALS if s["signal_type"] == stype]
    stats  = simulate_maker_strategy(subset)
    n = stats.get("n", 0)
    if n == 0:
        print(f"  {stype:<12}  no data")
        continue
    fe   = stats["fee_edge"]
    flag = "✅" if stats["positive"] else "❌"
    print(
        f"  {stype:<12}  "
        f"{stats['n_attempted']:>10}  "
        f"{stats['n_filled']:>8}  "
        f"{stats['win_rate']:>5.1%}  "
        f"{stats['avg_p']:>7.3f}  "
        f"{fe:>+8.1%}  "
        f"[{stats['ci_lo']:.3f},{stats['ci_hi']:.3f}]  {flag}"
    )

print()
print("Note: maker fee_edge = WR − avg_entry_p  (no fee subtracted — maker pays ₀).")
print("      Taker fee_edge = WR − avg_p − feeRate × avg_p × (1-avg_p).")
print()
print(f"  At avg_p≈0.40: maker advantage over taker ≈ feeRate × 0.40 × 0.60 = "
      f"{WEATHER_FEE_RATE * 0.40 * 0.60:.2%} extra edge per trade.")

## Section 9 — Kelly Sizing Stub

⚠️ **DO NOT RUN THIS SECTION IN LIVE TRADING WITHOUT ADDITIONAL VALIDATION.**

The Kelly Criterion recommends the fraction $f^*$ of bankroll to bet:

$$f^* = \frac{\text{fee\_edge}}{\sigma^2}$$

where $\sigma^2$ is the variance of returns.  We use **fractional Kelly** ($\frac{1}{4}$ Kelly) as a safety buffer, capped at 5% of bankroll per position.

This cell is a **sizing stub** — it calculates recommended bet sizes from backtest metrics but does **not** place any orders.  Wire it into `bot.py` after walk-forward validation passes the go-live gate.

In [ ]:
## ⚠️  SIZING STUB — outputs position sizes only; does NOT place orders  ⚠️

def fractional_kelly(fee_edge: float, std_dev: float,
                     fraction: float = KELLY_FRACTION,
                     max_bet: float  = KELLY_MAX_BET_PCT) -> float:
    """
    Return recommended fraction of bankroll to wager on one signal.

    Parameters
    ----------
    fee_edge : float
        Expected net edge after fees (e.g. 0.07 = 7%).
    std_dev  : float
        Standard deviation of per-trade returns.
    fraction : float
        Kelly multiplier; 0.25 = quarter-Kelly (conservative default).
    max_bet  : float
        Hard cap on bet size as a fraction of bankroll.

    Returns
    -------
    float  in [0, max_bet]
    """
    if std_dev <= 0 or fee_edge <= 0:
        return 0.0
    full_kelly = fee_edge / (std_dev ** 2)
    return min(full_kelly * fraction, max_bet)


# ── 9a. Compute Kelly sizes from Section 5 backtest stats ──────────────────

BANKROLL = 1_000.0  # $ — replace with live account equity before use

print("Kelly Sizing Recommendations  (stub — from Section 5 slippage-adjusted stats)")
print("=" * 75)
print(f"  Bankroll assumed : ${BANKROLL:,.2f}")
print(f"  Kelly fraction   : {KELLY_FRACTION:.0%}")
print(f"  Max bet cap      : {KELLY_MAX_BET_PCT:.0%} of bankroll")
print()
print(f"  {'Signal type':<12}  {'fee_edge':>9}  {'std_dev':>8}  "
      f"{'Kelly f':>8}  {'$ bet':>10}")
print("  " + "─" * 58)

for stype in ["whale", "consensus", "gated"]:
    _, slip = BACKTEST_RESULTS.get(stype, ({}, {}))
    n  = slip.get("n", 0)
    if n == 0:
        print(f"  {stype:<12}  no data")
        continue

    fe      = slip.get("fee_edge", 0.0)
    wr      = slip.get("win_rate", 0.0)
    avg_p   = slip.get("avg_p",    0.0)

    # Variance of per-trade P&L:
    # Win:  (1 - avg_p) with prob wr
    # Loss: (-avg_p)    with prob (1-wr)
    e_pl   = wr * (1 - avg_p) + (1 - wr) * (-avg_p)   # ≈ fee_edge
    e_pl2  = wr * (1 - avg_p)**2 + (1 - wr) * avg_p**2
    var_pl = e_pl2 - e_pl**2
    std_pl = math.sqrt(max(var_pl, 1e-9))

    kelly_f  = fractional_kelly(fe, std_pl)
    dollar_bet = kelly_f * BANKROLL

    print(
        f"  {stype:<12}  "
        f"{fe:>+8.1%}  "
        f"{std_pl:>8.4f}  "
        f"{kelly_f:>7.2%}  "
        f"${dollar_bet:>9.2f}"
    )

print()
print("  ⚠  Do not use these sizes until walk-forward gate passes (Section 7).")
print("  ⚠  Variance estimate is from historical data — live variance may differ.")